# NIG PROJECT

<p style="text-align: center;"><strong><span style="font-size: 20px;">Market Data Collection and Cleaning</span></strong></p>
<p><br></p>
<p>##Objective</p>
<p>The objective of this first stage is to collect, clean and validate S&amp;P 500 (SPX) European call option market data in order to build a reliable dataset for the calibration of the Normal Inverse Gaussian (NIG) model.</p>
<p>Observation date: 23 October 2026 &nbsp;</p>
<p>Underlying: S&amp;P 500 Index (SPX) &nbsp;</p>
<p>Initial maturity buckets: 5 &nbsp;</p>
<p>Initial moneyness range: 90% - 110% of spot</p>
<p><br></p>
<p>##Data Preparation Methodology</p>
<p>The cleaning process follows these steps:</p>
<p>1. Raw data import: Load the option data extracted from Bloomberg for five different expiration dates.</p>
<p>2. Moneyness filtering: Retain options with a strike-to-spot ratio between 0.90 and 1.10, focusing on options reasonably close to the money.</p>
<p>3. Market price construction: Calculate the midpoint price as the average of bid and ask prices, following the professor&apos;s approval. The last traded price is retained separately and is not used as the calibration target.</p>
<p>4. Market input preparation: Associate each option with its maturity, time to maturity, spot price, risk-free rate and dividend yield.</p>
<p>5. Data quality checks: Examine quote validity and preliminary no-arbitrage conditions.</p>
<p>6. Liquidity filtering: Retain options with positive trading volume and a relative bid-ask spread no greater than 2%.</p>
<p>7. Final dataset construction: Organise the selected contracts in a long-format table, with one observation per option contract.</p>
<p><br></p>
<p>## Final Dataset</p>
<p>The cleaning process reduces the initial universe of 916 option contracts to 318 liquid contracts, distributed across five maturities.</p>
<p>The resulting dataset contains the market prices and relevant inputs required for the subsequent calibration stage.</p>
<p>Output: `calibration_data` &nbsp;318 observations &times; 10 variables.</p>
<p>Note: The liquidity and data-quality filters improve the reliability of the calibration sample but do not, by themselves, guarantee that all retained prices satisfy every no-arbitrage condition.</p>

In [ ]:
import re
import pandas as pd
import numpy as np

### Data cleaning

#### Load data

In [ ]:
FILE_PATH = "SPX_options_data.xlsx"  # Path to the Excel file

# Define the column names for the DataFrame 
COLUMN_NAMES = [
    "Strike",
    "Ticker",
    "Bid",
    "Ask",
    "Last Trade Price",   #Bloomberg "Dern": Last traded price, not necessarily closing price
    "Implied Volatility",
    "Trading Volume",
]

# Read the Excel file into a dictionary of DataFrames, one for each sheet 
all_raw = pd.read_excel(
    FILE_PATH,
    sheet_name=None,
    skiprows=2,
    header = None,
    names=COLUMN_NAMES,
)

# Remove rows that are entirely empty
for sheet in all_raw.values():
    sheet.dropna(how="all", inplace = True)

In [ ]:
# Look at the shape and frist few rows of each DataFrame
for name, df in all_raw.items():
    print(name, df.shape)
    display(df.head())

### Clean data 
We apply the criteria explained in the report (only use options for which 0.9 <= Strike/S_0 <= 1.1 to get more stable results (see comment Flobelle))

In [ ]:
S_0 = 7623.68

# Filter each excel sheet to keep only 0.9 <= Strike / S_0 <= 1.1
df = {
    name: sheet.loc[
        sheet["Strike"].between(0.9 * S_0, 1.1 * S_0)
    ].reset_index(drop=True)
    for name, sheet in all_raw.items()
}

for name, sheet in df.items():
    print(name, sheet.shape)
    display(sheet.head())
    

In [ ]:
# ============================================================
# QUOTE VALIDATION AND MID-PRICE COMPUTATION
# ============================================================

# Check for zero Bid quotes
for sheet_name, sheet in df.items():

    bid_zero = sheet[sheet["Bid"] == 0]
    print(f"{sheet_name}: {len(bid_zero)} rows with Bid = 0")

    if not bid_zero.empty:
        display(bid_zero)

# Check for zero Ask quotes
for sheet_name, sheet in df.items():

    ask_zero = sheet[sheet["Ask"] == 0]
    print(f"{sheet_name}: {len(ask_zero)} rows with Ask = 0")

    if not ask_zero.empty:
        display(ask_zero)

# Compute the Mid-price as the average of Bid and Ask
for sheet in df.values():
    sheet["Mid"] = (sheet["Bid"] + sheet["Ask"]) / 2

    # Use Mid as the market price for calibration
    sheet["Market_Price"] = sheet["Mid"]

# Display the updated datasets
for name, sheet in df.items():
    print(name, sheet.shape)
    display(sheet.head())


<p style="text-align: center;"><strong>Additional Data Quality Checks</strong></p>
<p style="text-align: left;">This section performs additional data-quality checks on the selected option quotes. It focuses on missing values, invalid market quotes, crossed markets, duplicate contracts and trading-volume consistency. The purpose is to identify potential issues before applying liquidity filters and preparing the calibration dataset.</p>

In [ ]:
# ============================================================
# ADDITIONAL DATA QUALITY CHECKS
# ============================================================

quality_results = []

# Inspect each maturity separately
for maturity, sheet in df.items():

    # 1. Missing values in important columns
    missing_strike = sheet["Strike"].isna().sum()
    missing_ticker = sheet["Ticker"].isna().sum()
    missing_bid = sheet["Bid"].isna().sum()
    missing_ask = sheet["Ask"].isna().sum()
    missing_mid = sheet["Mid"].isna().sum()

    # 2. Invalid market quotes
    non_positive_bid = (sheet["Bid"] <= 0).sum()
    non_positive_ask = (sheet["Ask"] <= 0).sum()

    # 3. Crossed and locked markets
    crossed_quotes = (sheet["Ask"] < sheet["Bid"]).sum()
    locked_quotes = (sheet["Ask"] == sheet["Bid"]).sum()

    # 4. Duplicate contracts
    duplicate_tickers = sheet["Ticker"].duplicated().sum()

    # 5. Trading volume checks
    negative_volume = (sheet["Trading Volume"] < 0).sum()
    zero_volume = (sheet["Trading Volume"] == 0).sum()
    positive_volume = (sheet["Trading Volume"] > 0).sum()

    # 6. Implied volatility checks
    zero_iv = (sheet["Implied Volatility"] == 0).sum()
    negative_iv = (sheet["Implied Volatility"] < 0).sum()
    missing_iv = sheet["Implied Volatility"].isna().sum()

    # Store results
    quality_results.append({
        "Maturity": maturity,
        "Total Quotes": len(sheet),
        "Missing Strike": missing_strike,
        "Missing Ticker": missing_ticker,
        "Missing Bid": missing_bid,
        "Missing Ask": missing_ask,
        "Missing Mid": missing_mid,
        "Non-positive Bid": non_positive_bid,
        "Non-positive Ask": non_positive_ask,
        "Crossed Quotes": crossed_quotes,
        "Locked Quotes": locked_quotes,
        "Duplicate Tickers": duplicate_tickers,
        "Negative Volume": negative_volume,
        "Zero Volume": zero_volume,
        "Positive Volume": positive_volume,
        "Zero IV": zero_iv,
        "Negative IV": negative_iv,
        "Missing IV": missing_iv
    })

# Create a summary table
quality_report = pd.DataFrame(quality_results)

display(quality_report)

<p style="text-align: center;"><strong><span style="font-size: 20px;">Liquidity Analysis : Bid-Ask Spreads</span></strong></p>
<p style="text-align: left;">This section performs additional data-quality checks on the selected option quotes. It focuses on missing values, invalid market quotes, crossed markets, duplicate contracts and trading-volume consistency. The purpose is to identify potential issues before applying liquidity filters and preparing the calibration dataset.</p>

In [ ]:
# ============================================================
# LIQUIDITY ANALYSIS – BID-ASK SPREADS
# ============================================================

liquidity_results = []

for maturity, sheet in df.items():

    # Calculate absolute and relative bid-ask spreads
    sheet["Absolute_Spread"] = sheet["Ask"] - sheet["Bid"]

    sheet["Relative_Spread"] = (
        sheet["Absolute_Spread"] / sheet["Mid"]
    ) * 100

    # Summarise liquidity indicators for each maturity
    liquidity_results.append({
        "Maturity": maturity,
        "Number of Quotes": len(sheet),
        "Median Absolute Spread": sheet["Absolute_Spread"].median(),
        "Median Relative Spread (%)": sheet["Relative_Spread"].median(),
        "75th Percentile (%)": sheet["Relative_Spread"].quantile(0.75),
        "90th Percentile (%)": sheet["Relative_Spread"].quantile(0.90),
        "Positive Volume": (sheet["Trading Volume"] > 0).sum(),
        "Zero Volume": (sheet["Trading Volume"] == 0).sum()
    })

# Create the liquidity summary table
liquidity_report = pd.DataFrame(liquidity_results)

display(liquidity_report.round(4))

<p style="text-align: center;"><strong><span style="font-size: 20px;">Calibration Samples: Dataset Construction</span></strong></p>
<p style="text-align: left;">This section performs additional data-quality checks on the selected option quotes. It focuses on missing values, invalid market quotes, crossed markets, duplicate contracts and trading-volume consistency. The purpose is to identify potential issues before applying liquidity filters and preparing the calibration dataset.</p>

In [ ]:
# ============================================================
# CALIBRATION SAMPLES – DATASET CONSTRUCTION
# ============================================================

# Combine all maturities into one long-format dataset
valid_quotes = pd.concat(
    [
        sheet.assign(Maturity=maturity)
        for maturity, sheet in df.items()
    ],
    ignore_index=True
)

# Keep only observations with positive trading volume
liquid_quotes = valid_quotes.loc[
    valid_quotes["Trading Volume"] > 0
].copy()

# Reset indices
valid_quotes = valid_quotes.reset_index(drop=True)
liquid_quotes = liquid_quotes.reset_index(drop=True)

# Display sample sizes
print("All valid quotes:", len(valid_quotes))
print("Quotes with positive volume:", len(liquid_quotes))

# Compare the number of contracts by maturity
sample_comparison = pd.DataFrame({
    "All Valid Quotes": valid_quotes.groupby("Maturity").size(),
    "Positive Volume Quotes": liquid_quotes.groupby("Maturity").size()
})

display(sample_comparison)

<p style="text-align: center;"><strong><span style="font-size: 20px;">Maturity and Day : Count Convention Validation</span></strong></p>
<p style="text-align: left;">This section verifies that the maturity in days corresponds to the calendar difference between the observation date and each option expiry date. Time to maturity is expressed on an ACT/365 basis, while the money-market discount factor is calculated using an ACT/360 convention.</p>

In [ ]:
# ============================================================
# MATURITY AND DAY-COUNT CONVENTION VALIDATION
# ============================================================

from datetime import datetime

# Market observation date
valuation_date = datetime(2026, 10, 23)

# Option expiration dates, in the same order as the Excel sheets
expiry_dates = [
    datetime(2026, 11, 20),
    datetime(2026, 12, 18),
    datetime(2027, 1, 15),
    datetime(2027, 4, 16),
    datetime(2027, 9, 17)
]

# Calculate actual calendar days to maturity
actual_days = [
    (expiry - valuation_date).days
    for expiry in expiry_dates
]

# Compare the calculated days with the existing assumptions
maturity_check = pd.DataFrame({
    "Maturity": list(df.keys()),
    "Expiry Date": expiry_dates,
    "Expected Days": maturities_days,
    "Actual Calendar Days": actual_days
})

maturity_check["Match"] = (
    maturity_check["Expected Days"]
    == maturity_check["Actual Calendar Days"]
)

display(maturity_check)

## Market Inputs and European No-Arbitrage Lower Bound Validation

This section incorporates the market inputs required for option pricing, including
risk-free interest rates, dividend yield and time to maturity.

It also calculates the discount factors and checks whether observed call prices
satisfy the theoretical European lower bound, helping identify potential
inconsistencies in the market data before calibration.

In [ ]:
# Add the annual discount rate
q = 1.06 

# Add the annual risk-free rate for each maturity
r = [3.93, 4.03, 4.08, 4.2, 4.37]

In [ ]:
# Add the maturity in days for each sheet (same order as the sheets)
maturities_days = [28, 56, 84, 175, 329]

# Dividend yield: annual rate in percent
q_num = q / 100

# Check the arbitrage lower bound for each sheet
for (name, sheet), r_i, days in zip(df.items(), r, maturities_days):
    tau = days / 365                                   # time to maturity in years

    # SOFR OIS (maturity <= 1 year): simple interest, ACT/360
    discount = 1 / (1 + (r_i / 100) * days / 360)

    sheet["Days"] = days
    sheet["tau"] = tau
    sheet["Discount"] = discount
    sheet["r"] = -np.log(discount) / tau               # equivalent continuous rate (ACT/365)

    # European lower bound: max(0, S e^{-q tau} - K * D(tau))
    sheet["LB_European"] = np.maximum(
        0, S_0 * np.exp(-q_num * tau) - sheet["Strike"] * discount
    )
    # Bound as stated in Bakshi, Cao & Chen (additionally includes S - K)
    sheet["LB_BCC"] = np.maximum(sheet["LB_European"], S_0 - sheet["Strike"])

    sheet["Violates_European"] = (
    sheet["Market_Price"] < sheet["LB_European"]
)

    sheet["Violates_BCC"] = (
    sheet["Market_Price"] < sheet["LB_BCC"]
)

    print(f"{name}: {days} days, OIS = {r_i}% | "
          f"violations European: {sheet['Violates_European'].sum()}, "
          f"BCC: {sheet['Violates_BCC'].sum()}")

    violations = sheet[sheet["Violates_BCC"]]
    if not violations.empty:
        display(violations[["Strike", "Bid", "Ask", "Market_Price", "LB_European", "LB_BCC"]])

In [ ]:
# ============================================================
# BUILD THE FINAL LONG-FORMAT OPTION DATASET
# ============================================================

# Combine all maturity sheets into one dataset.
# Each row represents one individual call option contract.

option_data = []

for maturity, sheet in df.items():

    temp = sheet.copy()

    # Store the maturity associated with each option
    temp["Maturity"] = maturity

    option_data.append(temp)

# Combine all maturity DataFrames
options = pd.concat(option_data, ignore_index=True)

# Sort observations by maturity and strike
options = options.sort_values(
    ["Maturity", "Strike"]
).reset_index(drop=True)

# Display the resulting dataset
display(options.head(10))

print("Total number of option contracts:", len(options))
print("Number of maturities:", options["Maturity"].nunique())

In [ ]:
# ============================================================
# MONOTONICITY AND CONVEXITY VALIDATION
# ============================================================

# Analyse each maturity separately
convexity_results = []

for maturity, group in options.groupby("Maturity"):

    # Sort options by increasing strike
    group = group.sort_values("Strike").copy()

    # Extract strikes and market call prices
    strikes = group["Strike"].to_numpy()
    prices = group["Market_Price"].to_numpy()

    # Calculate price differences between consecutive strikes
    price_diff = np.diff(prices)

    # Calculate strike differences
    strike_diff = np.diff(strikes)

    # Calculate slopes between consecutive strikes
    slopes = price_diff / strike_diff

    # Monotonicity: call prices should not increase with strike
    monotonicity_violations = np.sum(price_diff > 0)

    # Convexity: slopes should be non-decreasing
    convexity_violations = np.sum(np.diff(slopes) < 0)

    convexity_results.append({
        "Maturity": maturity,
        "Number of Options": len(group),
        "Monotonicity Violations": monotonicity_violations,
        "Convexity Violations": convexity_violations
    })

# Display validation summary
convexity_report = pd.DataFrame(convexity_results)

display(convexity_report)

In [ ]:
# ============================================================
# IDENTIFY THE OPTION TRIPLETS THAT VIOLATE CONVEXITY
# ============================================================

convexity_details = []

for maturity, group in options.groupby("Maturity"):

    group = group.sort_values("Strike").reset_index(drop=True)

    strikes = group["Strike"].to_numpy()
    prices = group["Market_Price"].to_numpy()

    bids = group["Bid"].to_numpy()
    asks = group["Ask"].to_numpy()
    volumes = group["Trading Volume"].to_numpy()

    # Calculate slopes between consecutive strikes
    slopes = np.diff(prices) / np.diff(strikes)

    # Identify convexity violations
    for i in range(len(slopes) - 1):

        if slopes[i + 1] < slopes[i]:

            convexity_details.append({
                "Maturity": maturity,

                "K1": strikes[i],
                "K2": strikes[i + 1],
                "K3": strikes[i + 2],

                "C1": prices[i],
                "C2": prices[i + 1],
                "C3": prices[i + 2],

                "Slope 1": slopes[i],
                "Slope 2": slopes[i + 1],

                "Bid 1": bids[i],
                "Ask 1": asks[i],

                "Bid 2": bids[i + 1],
                "Ask 2": asks[i + 1],

                "Bid 3": bids[i + 2],
                "Ask 3": asks[i + 2],

                "Volume 1": volumes[i],
                "Volume 2": volumes[i + 1],
                "Volume 3": volumes[i + 2],

                "Violation Size": slopes[i] - slopes[i + 1]
            })

# Store the detailed results
convexity_details = pd.DataFrame(convexity_details)

# Display the largest violations first
convexity_details = convexity_details.sort_values(
    "Violation Size",
    ascending=False
)

display(convexity_details.head(15))

print("Total convexity violations:", len(convexity_details))

In [ ]:
# ============================================================
# BID-ASK INTERVAL CONVEXITY COMPATIBILITY
# ============================================================

interval_results = []

for maturity, group in options.groupby("Maturity"):

    group = group.sort_values("Strike").reset_index(drop=True)

    strikes = group["Strike"].to_numpy()

    bids = group["Bid"].to_numpy()
    asks = group["Ask"].to_numpy()

    # Consecutive strike intervals
    h1 = np.diff(strikes)[:-1]
    h2 = np.diff(strikes)[1:]

    # Maximum possible difference between consecutive slopes.
    # We choose prices within Bid-Ask intervals to make
    # the convexity condition as easy as possible to satisfy.

    max_slope_difference = (
        asks[2:] / h2
        + asks[:-2] / h1
        - bids[1:-1] * (1 / h1 + 1 / h2)
    )

    # If the maximum is non-negative, a convex price selection
    # is possible within the three Bid-Ask intervals.
    compatible = max_slope_difference >= 0

    interval_results.append({
        "Maturity": maturity,
        "Total Triplets": len(compatible),
        "Bid-Ask Compatible": int(compatible.sum()),
        "Not Compatible": int((~compatible).sum())
    })

interval_report = pd.DataFrame(interval_results)

display(interval_report)

print(
    "Total triplets:",
    interval_report["Total Triplets"].sum()
)

print(
    "Bid-Ask compatible:",
    interval_report["Bid-Ask Compatible"].sum()
)

print(
    "Not compatible:",
    interval_report["Not Compatible"].sum()
)

In [ ]:

# ============================================================
# COMMON STRIKES ACROSS MATURITIES
# Purpose: identify strike prices available for all maturities
# ============================================================

# List all available maturities
maturities = sorted(options["Maturity"].unique())

# Count the number of maturities available for each strike
strike_coverage = (
    options.groupby("Strike")["Maturity"]
    .nunique()
    .reset_index(name="Number_of_Maturities")
)

# Keep strikes available across all maturities
common_strikes = strike_coverage.loc[
    strike_coverage["Number_of_Maturities"] == len(maturities),
    "Strike"
]

# Summary by maturity
common_strike_summary = (
    options[options["Strike"].isin(common_strikes)]
    .groupby("Maturity")
    .agg(
        Number_of_Common_Strikes=("Strike", "nunique"),
        Number_of_Quotes=("Strike", "count")
    )
    .reset_index()
)

print("Number of maturities:", len(maturities))
print("Number of strikes common to all maturities:", len(common_strikes))

display(common_strike_summary)
display(
    strike_coverage[
        strike_coverage["Number_of_Maturities"] == len(maturities)
    ].sort_values("Strike")
)


In [ ]:
# Build the final option dataset in long format
# Each row represents one option contract

options_list = []

for maturity, sheet in df.items():

    # Use the Mid price as the market reference price
    price_column = (
        "Market_Price" if "Market_Price" in sheet.columns else "Mid"
    )

    # Copy the data to avoid modifying the original maturity sheets
    temp = sheet.copy()

    # Add maturity information
    temp["Maturity"] = maturity

    # Explicitly define the market price used for calibration
    temp["Market_Price"] = temp[price_column]

    options_list.append(temp)

# Combine all maturities into one dataset
options = pd.concat(options_list, ignore_index=True)

# Sort the dataset for easier inspection
options = options.sort_values(
    ["Maturity", "Strike"]
).reset_index(drop=True)

# Display the final dataset
display(options.head(10))

print("Total number of option contracts:", len(options))
print("Number of maturities:", options["Maturity"].nunique())
print("Missing market prices:", options["Market_Price"].isna().sum())

In [ ]:
# Final validation of the dataset used for NIG calibration

print("Total observations:", len(options))
print("Unique maturities:", options["Maturity"].nunique())
print("Missing market prices:", options["Market_Price"].isna().sum())
print("Non-positive market prices:", (options["Market_Price"] <= 0).sum())
print("Invalid bid-ask quotes:", (options["Ask"] < options["Bid"]).sum())
print("Zero-volume contracts:", (options["Trading Volume"] == 0).sum())

# Summary by maturity
display(
    options.groupby("Maturity").agg(
        Number_of_Contracts=("Strike", "count"),
        Positive_Volume=("Trading Volume", lambda x: (x > 0).sum()),
        Median_Market_Price=("Market_Price", "median"),
        Median_Relative_Spread=("Relative_Spread", "median")
    )
)

In [ ]:
# Relative bid-ask spread distribution

display(
    options.groupby("Maturity")["Relative_Spread"].agg(
        ["count", "median", "mean", "std",
         lambda x: x.quantile(0.75),
         lambda x: x.quantile(0.90),
         "max"]
    )
)

In [ ]:
# Assess the impact of a 2% relative bid-ask spread threshold

options["Spread_Above_2pct"] = options["Relative_Spread"] > 2

spread_filter_summary = (
    options.groupby("Maturity")
    .agg(
        Total_Contracts=("Ticker", "count"),
        Contracts_Above_2pct=("Spread_Above_2pct", "sum"),
        Median_Spread=("Relative_Spread", "median")
    )
)

spread_filter_summary["Contracts_Below_2pct"] = (
    spread_filter_summary["Total_Contracts"]
    - spread_filter_summary["Contracts_Above_2pct"]
)

display(spread_filter_summary)

print("Total contracts:", len(options))
print("Contracts above 2%:", options["Spread_Above_2pct"].sum())
print("Contracts at or below 2%:", (~options["Spread_Above_2pct"]).sum())

In [ ]:
# Combine spread and trading volume criteria

options["Liquidity_Pass"] = (
    (options["Relative_Spread"] <= 2) &
    (options["Trading Volume"] > 0)
)

liquidity_summary = (
    options.groupby("Maturity")
    .agg(
        Total_Contracts=("Ticker", "count"),
        Positive_Volume=("Trading Volume", lambda x: (x > 0).sum()),
        Acceptable_Spread=("Relative_Spread", lambda x: (x <= 2).sum()),
        Liquidity_Pass=("Liquidity_Pass", "sum")
    )
)

display(liquidity_summary)

print("Initial contracts:", len(options))
print("Contracts passing both criteria:", options["Liquidity_Pass"].sum())
print("Contracts excluded:", (~options["Liquidity_Pass"]).sum())

In [ ]:
# Evaluate strike coverage after liquidity filtering

liquid_options = options[options["Liquidity_Pass"]].copy()

liquid_options["Moneyness"] = (
    liquid_options["Strike"] / S_0
)

coverage_summary = (
    liquid_options.groupby("Maturity")
    .agg(
        Number_of_Contracts=("Ticker", "count"),
        Minimum_Strike=("Strike", "min"),
        Maximum_Strike=("Strike", "max"),
        Minimum_Moneyness=("Moneyness", "min"),
        Maximum_Moneyness=("Moneyness", "max"),
        Unique_Strikes=("Strike", "nunique")
    )
)

display(coverage_summary)

In [ ]:
# Analyse the distribution of moneyness across maturities

bins = [0.90, 0.95, 1.00, 1.05, 1.10]

labels = [
    "Deep OTM / Lower strikes",
    "OTM",
    "ATM to slightly ITM",
    "Higher strikes"
]

liquid_options["Moneyness_Bucket"] = pd.cut(
    liquid_options["Moneyness"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

moneyness_distribution = pd.crosstab(
    liquid_options["Maturity"],
    liquid_options["Moneyness_Bucket"]
)

display(moneyness_distribution)

In [ ]:
bins = [0.90, 0.95, 1.00, 1.05, 1.10]

labels = [
    "ITM (K/S0: 0.90–0.95)",
    "ITM / Near ATM (0.95–1.00)",
    "OTM / Near ATM (1.00–1.05)",
    "OTM (1.05–1.10)"
]

liquid_options["Moneyness_Bucket"] = pd.cut(
    liquid_options["Moneyness"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

moneyness_distribution = pd.crosstab(
    liquid_options["Maturity"],
    liquid_options["Moneyness_Bucket"]
)

display(moneyness_distribution)

In [ ]:
# ============================================
# FINAL DATASET FOR NIG CALIBRATION
# ============================================

# Keep only contracts passing the liquidity filters
calibration_data = liquid_options.copy()

# Market and model inputs
S_0 = 7623.68
q = 1.06 / 100  # Annual dividend yield

calibration_data["Spot"] = S_0
calibration_data["Dividend_Yield"] = q

# Use the bid-ask midpoint as the market price
calibration_data["Market_Price"] = calibration_data["Mid"]

# Sort observations by maturity and strike
calibration_data = calibration_data.sort_values(
    ["Maturity", "Strike"]
).reset_index(drop=True)

# Check the final dataset
print("Number of observations:", len(calibration_data))
print("Number of maturities:", calibration_data["Maturity"].nunique())
print("Missing values:", calibration_data.isna().sum().sum())

display(calibration_data.head())

In [ ]:
# Display the variables available for calibration

print("Available columns:")
print(calibration_data.columns.tolist())

print("\nDataset information:")
calibration_data.info()

print("\nSummary statistics:")
display(calibration_data.describe().T)

In [ ]:
# Select the variables required for option pricing and calibration

calibration_columns = [
    "Maturity", "Strike", "Market_Price", "Spot",
    "tau", "r", "Dividend_Yield",
    "Implied Volatility", "Trading Volume", "Relative_Spread"
]

calibration_data = calibration_data[calibration_columns].copy()

calibration_data = calibration_data.rename(
    columns={"Implied Volatility": "Implied_Volatility"}
)

calibration_data["Implied_Volatility"] /= 100

display(calibration_data.head())
print("Final dataset shape:", calibration_data.shape)